In [1]:
import tifffile as tiff
from pathlib import Path
import pandas as pd

Loading the data for preprocessing

In [55]:
DataPath = Path("..") / "Training data"

train_tabular = pd.read_csv(DataPath / "train_tabular.csv")
print(f"Tabular shape: {train_tabular.shape}")

train_tabular.drop(columns=["data_id"], inplace=True)

Tabular shape: (1024, 23)


Convert categorical columns to numeric

In [56]:
#Convert quarter_labels into a numeric format like 2020.25 for 2020-Q1, 2019.5 for 2019-Q2, etc.
quarter_mapping = {}
for row in train_tabular['quarter_label'].unique():
    label = row.split('-')
    numeric_label = float(label[0]) + (0.25 * int(label[1][1]) - 1)
    quarter_mapping[row] = numeric_label

train_tabular['quarter_label'] = train_tabular['quarter_label'].map(quarter_mapping)
#print(f"Unique quarter labels: {train_tabular['quarter_label'].nunique()}")
#print("Columns after processing:")
#print(train_tabular['quarter_label'].head())

#Convert yes/no columns to 1/0
yes_no_columns = ['developed_country', 'landlocked', 'access_to_airport', 'access_to_port', 'access_to_highway', 'access_to_railway', 'flood_risk_class']
for col in yes_no_columns:
    train_tabular[col] = train_tabular[col].map({'Yes': 1, 'No': 0}).astype('Int64')
    #print(f"Converted column {col} to numeric.")
    #print(train_tabular[col].head())

#Convert region_economic_classification to numeric codes
economic_map = {
    'Low income': 0,
    'Lower-middle income': 1,
    'Upper-middle income': 2,
    'High income': 3
}
train_tabular['region_economic_classification'] = train_tabular['region_economic_classification'].map(economic_map).astype('Int64')
#print(f"Converted 'region_economic_classification' to numeric codes.")
#print(train_tabular['region_economic_classification'].head())

risk_columns = ['seismic_hazard_zone', 'tropical_cyclone_wind_risk', 'tornadoes_wind_risk']
risk_map = {
    'Very Low': 0,
    'Low': 1,
    'Moderate': 2,
    'High': 3,
    'Very High': 4
}
for col in risk_columns:
    train_tabular[col] = train_tabular[col].map(risk_map).astype('Int64')
    #print(f"Converted column {col} to numeric.")
    #print(train_tabular[col].head())


nonnumeric_columns = train_tabular.select_dtypes(exclude=['number']).columns
print(f"Nonnumeric columns: {nonnumeric_columns.tolist()}")

print(train_tabular.columns)

Nonnumeric columns: ['geolocation_name', 'country', 'koppen_climate_zone', 'sentinel2_tiff_file_name', 'viirs_tiff_file_name']
Index(['geolocation_name', 'quarter_label', 'country', 'year',
       'deflated_gdp_usd', 'us_cpi', 'developed_country', 'landlocked',
       'region_economic_classification', 'access_to_airport', 'access_to_port',
       'access_to_highway', 'access_to_railway',
       'straight_distance_to_capital_km', 'seismic_hazard_zone',
       'flood_risk_class', 'tropical_cyclone_wind_risk', 'tornadoes_wind_risk',
       'koppen_climate_zone', 'sentinel2_tiff_file_name',
       'viirs_tiff_file_name', 'construction_cost_per_m2_usd'],
      dtype='str')


Since the data is split into two countries Japan and Philippines, the data is split to train two separate models. For each country to enhance model performance.

In [12]:
# Split the data based on country
philipines = pd.DataFrame()
japan = pd.DataFrame()

for row in train_tabular.itertuples(index=False):
    if row.country == "Japan":
        japan = pd.concat([japan, pd.DataFrame(data=[row])], ignore_index=True)
    elif row.country == "Philippines":
        philipines = pd.concat([philipines, pd.DataFrame(data=[row])], ignore_index=True)
    else:
        raise ValueError(f"Unknown country: {row.country}")

print(f"Philippines shape: {philipines.shape}")
print(f"Japan shape: {japan.shape}")

philipines.to_csv(DataPath / "train_tabular_philippines.csv", index=False)
japan.to_csv(DataPath / "train_tabular_japan.csv", index=False)

Philippines shape: (457, 23)
Japan shape: (567, 23)


Saving the data after preprocessing to use for model training.

In [11]:
SavePath = Path("..") / "Processed data"

#Drop colums with only one unique value
def drop_constant_columns(df):
    for col in df.columns:
        if df[col].nunique() == 1:
            df = df.drop(columns=[col])
    return df

philipines = drop_constant_columns(philipines)
japan = drop_constant_columns(japan)

print(f"Philippines shape: {philipines.shape}")
print(f"Japan shape: {japan.shape}")

Philippines shape: (457, 21)
Japan shape: (567, 19)
